## Structured Output

- Model can be requestd to provide their response in a formate matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output, including Pydantic models, JSON Schema, and custom output parsers.

### Pydantic

- Pydantic models provide the richest feature set with field validation, description, and nested structures.

In [39]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")  # Set your Groq API key here

model = init_chat_model(
    "groq:qwen/qwen3-32b"
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001591BF620D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001591C011B50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [40]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")

In [41]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001591BF620D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001591C011B50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [42]:
model.invoke("Provide the details of the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Inception is a 2010 science fiction action film directed by Christopher Nolan. The main actor is Leonardo DiCaprio, who plays the lead character, Dom Cobb. The story revolves around a technique called "inception," which is the process of planting an idea into someone\'s subconscious. \n\nThe plot is a bit complex. Dom Cobb is a thief who enters people\'s dreams to steal secrets. He\'s offered a chance to erase his criminal past by performing the reverse: planting an idea rather than stealing it. He has to go into the dreams of a target to implant an idea. There\'s a team involved, including Arthur (played by Joseph Gordon-Levitt) as Cobb\'s right-hand man, Ariadne (Elliot Page) who designs the dream, Eames (Tom Hardy) who can mimic people, and the target\'s daughter, Mal (Marion Cotillard), who\'s connected to Cobb\'s past.\n\nThe movie uses a lot of

In [43]:
model_with_structure.invoke("Provide details for the movie 'Inception'")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

#### Message output alongside Parsed Structured Output

In [44]:
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details for the movie 'Inception'")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie 'Inception'. Let me check the tools provided. There's a Movie function that requires title, year, director, and rating. I need to fill in those parameters. I know 'Inception' was directed by Christopher Nolan and released in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the exact year and rating. Yes, 2010 is correct, and the rating is 8.8. So I'll structure the tool call with those details.\n", 'tool_calls': [{'id': '7fagkfckk', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 224, 'total_tokens': 389, 'completion_time': 0.261680875, 'completion_tokens_details': {'reasoning_tokens': 117}, 'prompt_time': 0.009004403, 'prompt_tokens_details': None, 'queue_time': 0.051410537

#### Nested Structured Output

In [45]:
class Actor(BaseModel):
    name: str = Field(description="The name of the actor")
    role: str = Field(description="The role of the actor in the movie")

class MovieDetails(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")
    cast: list[Actor] = Field(description="List of cast members in the movie")
    genre: list[str] = Field(description="List of genres of the movie")
    budget: float | None = Field(description="Budget in millions USD")

In [46]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details for the movie 'Inception'")
response

MovieDetails(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genre=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

### TypedDict

- TypedDict is a lightweight alternative to Pydantic models, providing type hints for dictionaries without the overhead of full validation.

In [47]:
from typing_extensions import TypedDict, Annotated

class MovieDetailsTypedDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The release year of the movie"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The rating of the movie"]
    cast: Annotated[list[dict], "List of cast members in the movie"]
    genre: Annotated[list[str], "List of genres of the movie"]
    budget: Annotated[float | None, "Budget in millions USD"]

In [48]:
model_with_typeddict = model.with_structured_output(MovieDetailsTypedDict)
response = model_with_typeddict.invoke("Provide details for the movie Avengers")
response

{'title': 'Avengers'}

In [49]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClasses

- DataClasses are a built-in Python feature that allows for the creation of classes that primarily store data. They provide a simple way to define structured output without the need for additional libraries.

In [50]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model='groq:qwen/qwen3-32b',
    response_format=ContactInfo # Auto-select providerStrategy
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information : John Doe, john@example.com, 123-456-7890"
    }]
})

result

{'messages': [HumanMessage(content='Extract contact information : John Doe, john@example.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='00c1477a-a672-4e55-bd06-ed4f8eb1667f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given string. Let me look at the input: "John Doe, john@example.com, 123-456-7890". The function provided is ContactInfo, which requires name, email, and phone. \n\nFirst, I\'ll split the string by commas. The first part is the name, which is John Doe. The second part is the email, john@example.com. The third part is the phone number, 123-456-7890. \n\nI need to make sure each part is correctly assigned to the parameters. The required fields are name, email, and phone. All three are present here. No extra commas or missing data. \n\nI should check if the phone number is in the correct format. The user provided it as 123-456-7890, which is a standard US number. The f

In [51]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='123-456-7890')

In [52]:
## Same thing on typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfoTypedDict(TypedDict):
    """Contact information for a person."""
    name: str
    email: str
    phone: str

agent = create_agent(
    model='groq:qwen/qwen3-32b',
    response_format=ContactInfoTypedDict # Auto-select providerStrategy
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information : John Doe, john@example.com, 123-456-7890"
    }]
})

result['structured_response']

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '123-456-7890'}

In [53]:
# Same as dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfoDataClass:
    """Contact information for a person."""
    name: str
    email: str
    phone: str

agent = create_agent(
    model='groq:qwen/qwen3-32b',
    response_format=ContactInfoDataClass # Auto-select providerStrategy
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information : John Doe, john@example.com, 123-456-7890"
    }]
})

result['structured_response']

ContactInfoDataClass(name='John Doe', email='john@example.com', phone='123-456-7890')